# HDSoC Trigger-Time Consistency Playground

For each waveform, collect trigger times of contributing packets and inspect variance.
This answers: are trigger times consistent across packets in a waveform?

Packets are matched by **both channel and trigger timestamp**, so periodic batches cannot mix different triggers. Trigger-time consistency is a grouping invariant, not an independent measurement of trigger jitter.


In [ ]:
# Edit this path, then run the notebook from the top.
# Absolute paths and ~ are supported; relative paths start at the app directory.
DATA_FILE = ".artifacts/bank_format_verification/run121479/output.root"


In [ ]:
from pathlib import Path
import sys, os
# Works when Jupyter starts in the repository or a notebook subdirectory.
APP_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
                if (p / 'notebooks/notebook_support.py').is_file())
if str(APP_ROOT / 'notebooks') not in sys.path:
    sys.path.insert(0, str(APP_ROOT / 'notebooks'))
from notebook_support import (open_data, present, hit_waveform, hit_baseline,
    hit_amplitude, hit_peak, waveform_entries as find_waveform_entries,
    sampic_unit, integer, waveform_packets, sampic_event_bytes)
import ROOT
r = ROOT
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, Counter
root_file, tree, DATA_FILE = open_data('HDSoC', DATA_FILE)
f = file = root_file
plt.rcParams.update({'figure.figsize': (12, 5), 'axes.grid': True, 'grid.alpha': 0.25})


In [ ]:
# Setup is performed above.


In [ ]:
# Setup is performed above.


In [ ]:
def waveform_trigger_times(evt, wf):
    return [int(p.trigger_time) for p in waveform_packets(evt, wf)]


In [ ]:
# Inspect one waveform
event_index = 0
waveform_index = 0

tree.GetEntry(event_index)
evt = tree.nalu_event
wfs = list(evt.waveforms.waveforms)

if waveform_index >= len(wfs):
    raise IndexError(f"waveform_index={waveform_index} out of range for event {event_index}")

wf = wfs[waveform_index]
tts = waveform_trigger_times(evt, wf)
print(f"Event {event_index}, waveform {waveform_index}, channel {integer(wf.channel_num)}")
print("Trigger times:", tts)
print("Variance:", float(np.var(tts)) if tts else float('nan'))
print("Range (max-min):", (max(tts) - min(tts)) if tts else -1)


In [ ]:
# Per-waveform trigger-time variance and spread, grouped by channel
variance_by_channel = defaultdict(list)
range_by_channel = defaultdict(list)

for entry in range(tree.GetEntries()):
    tree.GetEntry(entry)
    evt = tree.nalu_event

    for wf in evt.waveforms.waveforms:
        ch = integer(wf.channel_num)
        tts = waveform_trigger_times(evt, wf)
        if len(tts) == 0:
            continue
        arr = np.asarray(tts, dtype=np.float64)
        variance_by_channel[ch].append(float(np.var(arr)))
        range_by_channel[ch].append(float(np.max(arr) - np.min(arr)))

channels = sorted(variance_by_channel.keys())

fig, axes = plt.subplots(len(channels), 2, figsize=(12, 3.5 * len(channels)))
if len(channels) == 1:
    axes = np.array([axes])

for row, ch in enumerate(channels):
    ax_var = axes[row, 0]
    ax_rng = axes[row, 1]

    v = variance_by_channel[ch]
    rvals = range_by_channel[ch]

    ax_var.hist(v, bins=50, edgecolor="black")
    ax_var.set_title(f"Channel {ch}: per-waveform trigger_time variance")
    ax_var.set_xlabel("Variance")
    ax_var.set_ylabel("Waveform count")

    ax_rng.hist(rvals, bins=range(int(min(rvals)), int(max(rvals)) + 2), edgecolor="black", align="left")
    ax_rng.set_title(f"Channel {ch}: per-waveform trigger_time range")
    ax_rng.set_xlabel("max(trigger_time)-min(trigger_time)")
    ax_rng.set_ylabel("Waveform count")

plt.tight_layout()
plt.show()


In [ ]:
# Quick numeric summary
for ch in sorted(variance_by_channel.keys()):
    v = np.asarray(variance_by_channel[ch], dtype=np.float64)
    rvals = np.asarray(range_by_channel[ch], dtype=np.float64)
    print(f"Channel {ch}:")
    print(f"  waveforms: {len(v)}")
    print(f"  variance median={np.median(v):.3f}, max={np.max(v):.3f}")
    print(f"  range median={np.median(rvals):.3f}, max={np.max(rvals):.3f}")
    print(f"  fraction exact-consistent (range==0): {np.mean(rvals == 0):.3f}")


In [ ]:
f.Close()
